# TP Olist : découverte, audit et nettoyage des données

Dans ce notebook, j'essaie de comprendre les données Olist (un site de vente en ligne brésilien), de repérer ce qui ne va pas, puis de les nettoyer.

Mon plan :
1. **Charger** les 9 fichiers CSV
2. **Comprendre** ce qu'il y a dedans et comment les fichiers sont reliés
3. **Faire l'audit** : je compte les problèmes, mais je ne modifie encore rien
4. **Nettoyer** les problèmes trouvés
5. **Regrouper** les tables qui ont plusieurs lignes par commande
6. **Assembler** une table finale avec une ligne par commande

Mes explications plus détaillées sont dans `CARNET.md`.

# 0. Chargement des fichiers

Je lis les 9 CSV avec `pd.read_csv`. Pour les colonnes qui contiennent des dates, j'utilise `parse_dates` : sinon pandas les lit comme du texte et je ne peux pas les comparer entre elles.

In [8]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)   # pour voir toutes les colonnes
DATA = Path("/workspace/donnees/olist")    # dossier où sont les CSV

def lire(nom, dates=None):
    return pd.read_csv(DATA / nom, parse_dates=dates)

customers   = lire("olist_customers_dataset.csv")
geo         = lire("olist_geolocation_dataset.csv")
items       = lire("olist_order_items_dataset.csv", ["shipping_limit_date"])
payments    = lire("olist_order_payments_dataset.csv")
reviews     = lire("olist_order_reviews_dataset.csv", ["review_creation_date", "review_answer_timestamp"])
orders      = lire("olist_orders_dataset.csv", ["order_purchase_timestamp", "order_approved_at",
                                                "order_delivered_carrier_date", "order_delivered_customer_date",
                                                "order_estimated_delivery_date"])
products    = lire("olist_products_dataset.csv")
sellers     = lire("olist_sellers_dataset.csv")
translation = lire("product_category_name_translation.csv")

# Je range tout dans un dictionnaire pour pouvoir faire des boucles sur les 9 tables
tables = {"customers": customers, "geolocation": geo, "order_items": items, "order_payments": payments,
          "order_reviews": reviews, "orders": orders, "products": products,
          "sellers": sellers, "translation": translation}

# 1. Comprendre les données

## 1.1 Taille et colonnes de chaque fichier

In [9]:
pd.DataFrame({nom: {"lignes": len(df), "colonnes": df.shape[1],
                    "liste des colonnes": ", ".join(df.columns)}
              for nom, df in tables.items()}).T

,lignes,colonnes,liste des colonnes
customers,99441,5,"customer_id, customer_unique_id, customer_zip_..."
geolocation,1000163,5,"geolocation_zip_code_prefix, geolocation_lat, ..."
order_items,112650,7,"order_id, order_item_id, product_id, seller_id..."
order_payments,103886,5,"order_id, payment_sequential, payment_type, pa..."
order_reviews,99224,7,"review_id, order_id, review_score, review_comm..."
orders,99441,8,"order_id, customer_id, order_status, order_pur..."
products,32951,9,"product_id, product_category_name, product_nam..."
sellers,3095,4,"seller_id, seller_zip_code_prefix, seller_city..."
translation,71,2,"product_category_name, product_category_name_e..."


Je regarde à quoi ressemble la table principale, `orders` :

In [10]:
orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04


## 1.2 Comment les tables sont reliées

En lisant les noms de colonnes, je vois que certaines reviennent dans plusieurs fichiers (`order_id`, `customer_id`, `product_id`…). Ce sont les **clés** qui relient les tables. J'ai fait ce schéma :

```
customers ──customer_id── orders ──order_id── order_items ──product_id── products ──category── translation
    │                        │                     └──seller_id── sellers
    └─zip_code_prefix─┐      ├──order_id── order_payments
                      │      └──order_id── order_reviews
                  geolocation (zip_code_prefix)
```

La table au centre, c'est `orders` (les commandes).

Ensuite je vérifie, pour chaque clé, s'il y a **une seule ligne** par valeur ou **plusieurs**. Par exemple : est-ce qu'une commande apparaît une seule fois dans `order_items`, ou plusieurs fois ?

In [11]:
def verifier_cle(nom, df, col):
    return {"table": nom, "clé": col,
            "valeurs différentes": df[col].nunique(), "lignes": len(df),
            "résultat": "une ligne par clé" if df[col].is_unique else "plusieurs lignes par clé"}

pd.DataFrame([
    verifier_cle("orders", orders, "order_id"),
    verifier_cle("orders", orders, "customer_id"),
    verifier_cle("customers", customers, "customer_id"),
    verifier_cle("customers", customers, "customer_unique_id"),
    verifier_cle("order_items", items, "order_id"),
    verifier_cle("order_payments", payments, "order_id"),
    verifier_cle("order_reviews", reviews, "order_id"),
    verifier_cle("products", products, "product_id"),
    verifier_cle("sellers", sellers, "seller_id"),
    verifier_cle("geolocation", geo, "geolocation_zip_code_prefix"),
])

,table,clé,valeurs différentes,lignes,résultat
0,orders,order_id,99441,99441,une ligne par clé
1,orders,customer_id,99441,99441,une ligne par clé
2,customers,customer_id,99441,99441,une ligne par clé
3,customers,customer_unique_id,96096,99441,plusieurs lignes par clé
4,order_items,order_id,98666,112650,plusieurs lignes par clé
5,order_payments,order_id,99440,103886,plusieurs lignes par clé
6,order_reviews,order_id,98673,99224,plusieurs lignes par clé
7,products,product_id,32951,32951,une ligne par clé
8,sellers,seller_id,3095,3095,une ligne par clé
9,geolocation,geolocation_zip_code_prefix,19015,1000163,plusieurs lignes par clé


Ce que j'en retiens :
- Il y a autant de `customer_id` que de commandes. En fait, un nouveau `customer_id` est créé **à chaque commande**. Pour retrouver une même personne qui commande plusieurs fois, il faut utiliser `customer_unique_id`.
- `order_items`, `order_payments`, `order_reviews` et `geolocation` ont **plusieurs lignes pour une même clé**. Si je les fusionne directement avec `orders`, des commandes vont se retrouver en double. Il faudra d'abord les regrouper (partie 4).

# 2. Audit de qualité

L'idée de cette partie : **avant de nettoyer, je mesure**. Je compte les problèmes, mais je ne touche pas encore aux données.

Je cherche 5 types de problèmes :
1. les valeurs manquantes
2. les doublons
3. les dates incohérentes
4. les incohérences entre fichiers
5. les valeurs aberrantes

Pour ne rien oublier, je note chaque problème trouvé dans une liste `audit` avec une petite fonction `note()`. À la fin (2.6), j'affiche tout dans un seul tableau.

In [12]:
audit = []

def note(fichier, famille, probleme, n, total):
    """Ajoute un problème à la liste et renvoie le nombre de lignes concernées."""
    audit.append({"fichier": fichier, "famille": famille, "problème": probleme,
                  "lignes": int(n), "%": round(100 * n / total, 2)})
    return int(n)

## 2.1 Valeurs manquantes

J'utilise `isna().sum()` qui compte les cases vides dans chaque colonne.

In [13]:
for nom, df in tables.items():
    for col, n in df.isna().sum().items():
        if n > 0:
            note(nom, "Valeurs manquantes", f"{col} vide", n, len(df))

pd.DataFrame([a for a in audit if a["famille"] == "Valeurs manquantes"])

,fichier,famille,problème,lignes,%
0,order_reviews,Valeurs manquantes,review_comment_title vide,87656,88.34
1,order_reviews,Valeurs manquantes,review_comment_message vide,58247,58.70
2,orders,Valeurs manquantes,order_approved_at vide,160,0.16
3,orders,Valeurs manquantes,order_delivered_carrier_date vide,1783,1.79
4,orders,Valeurs manquantes,order_delivered_customer_date vide,2965,2.98
5,products,Valeurs manquantes,product_category_name vide,610,1.85
6,products,Valeurs manquantes,product_name_lenght vide,610,1.85
7,products,Valeurs manquantes,product_description_lenght vide,610,1.85
8,products,Valeurs manquantes,product_photos_qty vide,610,1.85
9,products,Valeurs manquantes,product_weight_g vide,2,0.01


Dans `orders`, il manque des dates de livraison. Je me demande si c'est normal : une commande annulée ou pas encore livrée n'a logiquement pas de date de livraison. Je vérifie en croisant avec le statut :

In [14]:
pd.crosstab(orders.order_status, orders.order_delivered_customer_date.isna(),
            colnames=["date de livraison vide ?"])

date de livraison vide ?,False,True
order_status,,
approved,0,2
canceled,6,619
created,0,5
delivered,96470,8
invoiced,0,314
processing,0,301
shipped,0,1107
unavailable,0,609


## 2.2 Doublons

Je cherche deux sortes de doublons :
- les lignes **complètement identiques** (avec `duplicated()`),
- les **identifiants qui devraient être uniques** mais qui apparaissent plusieurs fois.

In [15]:
# Lignes identiques dans chaque table
for nom, df in tables.items():
    n = df.duplicated().sum()
    if n > 0:
        note(nom, "Doublons", "lignes complètement identiques", n, len(df))

# Identifiants répétés
note("order_reviews", "Doublons", "review_id qui apparaît plusieurs fois", reviews.review_id.duplicated().sum(), len(reviews))
note("order_reviews", "Doublons", "commande avec plusieurs avis", reviews.order_id.duplicated().sum(), len(reviews))
note("geolocation", "Doublons", "code postal avec plusieurs coordonnées GPS",
     geo.drop_duplicates().geolocation_zip_code_prefix.duplicated().sum(), len(geo))

pd.DataFrame([a for a in audit if a["famille"] == "Doublons"])

,fichier,famille,problème,lignes,%
0,geolocation,Doublons,lignes complètement identiques,261831,26.18
1,order_reviews,Doublons,review_id qui apparaît plusieurs fois,814,0.82
2,order_reviews,Doublons,commande avec plusieurs avis,551,0.56
3,geolocation,Doublons,code postal avec plusieurs coordonnées GPS,719317,71.92


## 2.3 Cohérence des dates

Une commande suit normalement cet ordre :

**achat → validation du paiement → remise au transporteur → livraison au client**

Donc une date ne devrait jamais être avant la précédente. Je compte les commandes où ce n'est pas le cas.

Remarque : quand une des deux dates est vide, la comparaison donne `False`, donc ces lignes ne sont pas comptées comme des erreurs.

In [16]:
o = orders   # nom plus court

regles = {
    "validation avant l'achat":         o.order_approved_at < o.order_purchase_timestamp,
    "transporteur avant la validation": o.order_delivered_carrier_date < o.order_approved_at,
    "transporteur avant l'achat":       o.order_delivered_carrier_date < o.order_purchase_timestamp,
    "livraison avant le transporteur":  o.order_delivered_customer_date < o.order_delivered_carrier_date,
    "livraison avant l'achat":          o.order_delivered_customer_date < o.order_purchase_timestamp,
    "livraison prévue avant l'achat":   o.order_estimated_delivery_date < o.order_purchase_timestamp,
}
for probleme, erreurs in regles.items():
    note("orders", "Cohérence temporelle", probleme, erreurs.sum(), len(o))

# Pour les avis : on ne peut pas répondre à un avis avant qu'il soit créé
note("order_reviews", "Cohérence temporelle", "réponse avant la création de l'avis",
     (reviews.review_answer_timestamp < reviews.review_creation_date).sum(), len(reviews))

pd.DataFrame([a for a in audit if a["famille"] == "Cohérence temporelle"])

,fichier,famille,problème,lignes,%
0,orders,Cohérence temporelle,validation avant l'achat,0,0.00
1,orders,Cohérence temporelle,transporteur avant la validation,1359,1.37
2,orders,Cohérence temporelle,transporteur avant l'achat,166,0.17
3,orders,Cohérence temporelle,livraison avant le transporteur,23,0.02
4,orders,Cohérence temporelle,livraison avant l'achat,0,0.00
5,orders,Cohérence temporelle,livraison prévue avant l'achat,0,0.00
6,order_reviews,Cohérence temporelle,réponse avant la création de l'avis,0,0.00


Je regarde aussi sur quelle période vont les données, et combien il y a de commandes par mois. S'il y a très peu de commandes au début ou à la fin, c'est sûrement que ces mois ne sont pas complets.

In [17]:
print("Première commande :", o.order_purchase_timestamp.min())
print("Dernière commande :", o.order_purchase_timestamp.max())
o.order_purchase_timestamp.dt.to_period("M").value_counts().sort_index()

Première commande : 2016-09-04 21:15:19
Dernière commande : 2018-10-17 17:30:18


order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

## 2.4 Cohérence entre les fichiers

Si un `order_id` est dans `order_items`, il devrait aussi exister dans `orders`. Sinon, l'article appartient à une commande « fantôme ». J'appelle ça des **orphelins**.

Pour les trouver, j'utilise `isin()` : il me dit si chaque valeur existe dans l'autre table.

In [19]:
def orphelins(fichier, probleme, enfant, col_enfant, parent, col_parent):
    absents = ~enfant[col_enfant].isin(parent[col_parent])   
    return note(fichier, "Cohérence entre fichiers", probleme, absents.sum(), len(enfant))

orphelins("order_items",    "order_id inconnu dans orders",          items, "order_id", orders, "order_id")
orphelins("order_payments", "order_id inconnu dans orders",          payments, "order_id", orders, "order_id")
orphelins("order_reviews",  "order_id inconnu dans orders",          reviews, "order_id", orders, "order_id")
orphelins("orders",         "customer_id inconnu dans customers",    orders, "customer_id", customers, "customer_id")
orphelins("order_items",    "product_id inconnu dans products",      items, "product_id", products, "product_id")
orphelins("order_items",    "seller_id inconnu dans sellers",        items, "seller_id", sellers, "seller_id")
orphelins("customers",      "code postal inconnu dans geolocation",  customers, "customer_zip_code_prefix", geo, "geolocation_zip_code_prefix")
orphelins("sellers",        "code postal inconnu dans geolocation",  sellers, "seller_zip_code_prefix", geo, "geolocation_zip_code_prefix")

# Dans l'autre sens : des commandes sans rien derrière
orphelins("orders", "commande sans aucun article", orders, "order_id", items, "order_id")
orphelins("orders", "commande sans paiement",      orders, "order_id", payments, "order_id")
orphelins("orders", "commande sans avis",          orders, "order_id", reviews, "order_id")

# Catégories de produits qui n'ont pas de traduction
categories = products.product_category_name.dropna()
note("products", "Cohérence entre fichiers", "catégorie sans traduction",
     (~categories.isin(translation.product_category_name)).sum(), len(products))
print("Catégories sans traduction :", set(categories) - set(translation.product_category_name))

Catégories sans traduction : {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}


Deux autres vérifications :
- est-ce que le **statut** de la commande correspond à ses dates ? (une commande « livrée » doit avoir une date de livraison)
- est-ce que le **montant payé** correspond bien au prix des articles + les frais de port ?

In [20]:
note("orders", "Cohérence entre fichiers", "statut 'delivered' mais pas de date de livraison",
     ((o.order_status == "delivered") & o.order_delivered_customer_date.isna()).sum(), len(o))
note("orders", "Cohérence entre fichiers", "statut 'canceled' mais une date de livraison",
     ((o.order_status == "canceled") & o.order_delivered_customer_date.notna()).sum(), len(o))

# Total des articles (prix + port) par commande, comparé au total payé
total_articles = items.groupby("order_id")[["price", "freight_value"]].sum().sum(axis=1)
total_paye     = payments.groupby("order_id").payment_value.sum()
ecart = (total_paye - total_articles).dropna().abs()

note("order_payments", "Cohérence entre fichiers", "montant payé différent de prix + port (écart > 1 R$)",
     (ecart > 1).sum(), len(ecart))
ecart[ecart > 1].describe()

count    249.000000
mean      13.101727
std       18.310444
min        1.010000
25%        3.960000
50%        7.790000
75%       14.990000
max      182.810000
dtype: float64

## 2.5 Valeurs aberrantes

Une valeur aberrante, c'est une valeur impossible (un prix négatif, un poids de 0 g…) ou très éloignée des autres.

Je commence par `describe()`, qui donne le minimum, le maximum, la moyenne et les quartiles de chaque colonne. Ça permet de repérer rapidement les valeurs bizarres.

In [21]:
items[["price", "freight_value"]].describe()

,price,freight_value
count,112650.000000,112650.000000
mean,120.653739,19.990320
std,183.633928,15.806405
min,0.850000,0.000000
25%,39.900000,13.080000
50%,74.990000,16.260000
75%,134.900000,21.150000
max,6735.000000,409.680000


In [22]:
payments[["payment_value", "payment_installments"]].describe()

,payment_value,payment_installments
count,103886.000000,103886.000000
mean,154.100380,2.853349
std,217.494064,2.687051
min,0.000000,0.000000
25%,56.790000,1.000000
50%,100.000000,1.000000
75%,171.837500,4.000000
max,13664.080000,24.000000


In [23]:
products.select_dtypes("number").describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


Pour les colonnes de texte, j'utilise `value_counts()` pour voir toutes les valeurs possibles :

In [24]:
payments.payment_type.value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Je compte maintenant les valeurs qui me semblent impossibles :

In [25]:
note("order_items",    "Valeurs aberrantes", "prix ≤ 0",                         (items.price <= 0).sum(), len(items))
note("order_items",    "Valeurs aberrantes", "frais de port = 0",                (items.freight_value == 0).sum(), len(items))
note("order_payments", "Valeurs aberrantes", "paiement de 0",                    (payments.payment_value == 0).sum(), len(payments))
note("order_payments", "Valeurs aberrantes", "paiement en 0 fois",               (payments.payment_installments == 0).sum(), len(payments))
note("order_payments", "Valeurs aberrantes", "type de paiement 'not_defined'",   (payments.payment_type == "not_defined").sum(), len(payments))
note("products",       "Valeurs aberrantes", "poids de 0 g",                     (products.product_weight_g == 0).sum(), len(products))

# Le Brésil est à peu près entre -34 et 6 de latitude, et entre -74 et -34 de longitude
hors_bresil = ~geo.geolocation_lat.between(-34, 6) | ~geo.geolocation_lng.between(-74, -34)
note("geolocation", "Valeurs aberrantes", "coordonnées GPS en dehors du Brésil", hors_bresil.sum(), len(geo))

42

Pour les valeurs **très grandes**, j'utilise la méthode de l'écart interquartile (IQR) vue en stats :
- Q1 = la valeur sous laquelle se trouvent 25 % des données, Q3 = celle sous laquelle se trouvent 75 %
- IQR = Q3 − Q1
- une valeur est considérée comme extrême si elle dépasse **Q3 + 1,5 × IQR**

Attention : extrême ne veut pas dire faux. Un frigo coûte plus cher qu'un stylo, c'est normal.

In [26]:
def valeurs_extremes(fichier, df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    seuil = q3 + 1.5 * (q3 - q1)
    n = note(fichier, "Valeurs aberrantes", f"{col} > {seuil:.0f} (méthode IQR)", (df[col] > seuil).sum(), len(df))
    print(f"{col} : {n} valeurs au-dessus de {seuil:.1f} (le max est {df[col].max()})")

valeurs_extremes("order_items", items, "price")
valeurs_extremes("order_items", items, "freight_value")
valeurs_extremes("order_payments", payments, "payment_value")
valeurs_extremes("products", products, "product_weight_g")

price : 8427 valeurs au-dessus de 277.4 (le max est 6735.0)
freight_value : 11613 valeurs au-dessus de 33.3 (le max est 409.68)
payment_value : 7981 valeurs au-dessus de 344.4 (le max est 13664.08)
product_weight_g : 4551 valeurs au-dessus de 4300.0 (le max est 40425.0)


Dernière chose : les noms de villes. J'ai l'impression qu'une même ville peut être écrite de plusieurs façons (avec ou sans accent, en majuscules…). Je compare le nombre de villes avant et après avoir tout mis en minuscules sans accents :

In [27]:
villes = geo.geolocation_city
villes_simplifiees = (villes.str.lower()
                      .str.normalize("NFKD").str.encode("ascii", "ignore").str.decode("ascii"))
print("Nombre de villes différentes           :", villes.nunique())
print("Après minuscules et sans les accents  :", villes_simplifiees.nunique())

Nombre de villes différentes           : 8011
Après minuscules et sans les accents  : 5968


## 2.6 Bilan de l'audit

Voici tous les problèmes trouvés, rangés par fichier. Je m'en sers pour décider quoi nettoyer et pour remplir `CARNET.md`.

In [28]:
bilan = (pd.DataFrame(audit)
         .query("lignes > 0")                      # je garde seulement les vrais problèmes
         .sort_values(["fichier", "famille", "lignes"], ascending=[True, True, False])
         .reset_index(drop=True))
bilan

,fichier,famille,problème,lignes,%
0,customers,Cohérence entre fichiers,code postal inconnu dans geolocation,278,0.28
1,customers,Cohérence entre fichiers,code postal inconnu dans geolocation,278,0.28
2,geolocation,Doublons,code postal avec plusieurs coordonnées GPS,719317,71.92
3,geolocation,Doublons,lignes complètement identiques,261831,26.18
4,geolocation,Valeurs aberrantes,coordonnées GPS en dehors du Brésil,42,0.00
5,order_items,Valeurs aberrantes,freight_value > 33 (méthode IQR),11613,10.31
6,order_items,Valeurs aberrantes,price > 277 (méthode IQR),8427,7.48
7,order_items,Valeurs aberrantes,frais de port = 0,383,0.34
8,order_payments,Cohérence entre fichiers,montant payé différent de prix + port (écart >...,249,0.25
9,order_payments,Valeurs aberrantes,payment_value > 344 (méthode IQR),7981,7.68


Le même bilan résumé : combien de problèmes de chaque type par fichier.

In [29]:
bilan.pivot_table(index="fichier", columns="famille", values="lignes", aggfunc="count", fill_value=0)

famille,Cohérence entre fichiers,Cohérence temporelle,Doublons,Valeurs aberrantes,Valeurs manquantes
fichier,,,,,
customers,2,0,0,0,0
geolocation,0,0,2,1,0
order_items,0,0,0,3,0
order_payments,1,0,0,4,0
order_reviews,0,0,2,0,2
orders,8,3,0,0,3
products,2,0,0,2,8
sellers,2,0,0,0,0


# 3. Nettoyage

Maintenant que je sais ce qui ne va pas, je corrige. Je travaille sur des **copies** (`.copy()`) pour garder les données d'origine intactes, au cas où je me tromperais.

In [30]:
orders_propre   = orders.copy()
products_propre = products.copy()
geo_propre      = geo.copy()
reviews_propre  = reviews.copy()

## 3.1 orders : calculer les délais

À partir des dates, je calcule deux nouvelles colonnes utiles pour l'analyse :
- le **délai de livraison** en jours (entre l'achat et la livraison)
- le **retard** en jours (entre la date prévue et la date réelle ; s'il est positif, la commande est en retard)

In [31]:
orders_propre["delai_livraison_jours"] = (orders_propre.order_delivered_customer_date
                                          - orders_propre.order_purchase_timestamp).dt.days
orders_propre["retard_jours"] = (orders_propre.order_delivered_customer_date
                                 - orders_propre.order_estimated_delivery_date).dt.days
orders_propre["en_retard"] = orders_propre["retard_jours"] > 0

orders_propre[["delai_livraison_jours", "retard_jours"]].describe()

,delai_livraison_jours,retard_jours
count,96476.000000,96476.000000
mean,12.094086,-11.876881
std,9.551746,10.183854
min,0.000000,-147.000000
25%,6.000000,-17.000000
50%,10.000000,-12.000000
75%,15.000000,-7.000000
max,209.000000,188.000000


## 3.2 products : colonnes, catégories vides et traduction

- Deux colonnes ont une faute d'orthographe (`lenght` au lieu de `length`), je les renomme.
- 610 produits n'ont pas de catégorie : je ne veux pas les supprimer car ils ont été vendus, donc je mets `"inconnue"`.
- J'ajoute le nom de la catégorie en anglais grâce à la table `translation`.

In [32]:
products_propre = products_propre.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})
products_propre["product_category_name"] = products_propre["product_category_name"].fillna("inconnue")

products_propre = products_propre.merge(translation, on="product_category_name", how="left")
# Si pas de traduction, je garde le nom portugais
products_propre["product_category_name_english"] = (products_propre["product_category_name_english"]
                                                    .fillna(products_propre["product_category_name"]))
products_propre.isna().sum()

product_id                         0
product_category_name              0
product_name_length              610
product_description_length       610
product_photos_qty               610
product_weight_g                   2
product_length_cm                  2
product_height_cm                  2
product_width_cm                   2
product_category_name_english      0
dtype: int64

## 3.3 geolocation : doublons et coordonnées fausses

Je supprime les lignes identiques, puis les points qui ne sont pas au Brésil.

In [33]:
print("Au départ                   :", len(geo_propre))
geo_propre = geo_propre.drop_duplicates()
print("Sans les doublons            :", len(geo_propre))
geo_propre = geo_propre[geo_propre.geolocation_lat.between(-34, 6) & geo_propre.geolocation_lng.between(-74, -34)]
print("Sans les points hors Brésil  :", len(geo_propre))

Au départ                   : 1000163
Sans les doublons            : 738332
Sans les points hors Brésil  : 738299


## 3.4 reviews : les commandes avec plusieurs avis

Je regarde à quoi ressemblent ces cas avant de décider quoi faire :

In [34]:
reviews_propre[reviews_propre.order_id.duplicated(keep=False)].sort_values("order_id").head(6)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29,2017-08-30 01:59:12
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25,2017-08-29 21:45:57
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:47


# 4. Regrouper (agrégations)

Problème : certaines tables ont **plusieurs lignes pour une seule commande** (plusieurs articles, plusieurs paiements…). Je ne veux pas supprimer ces lignes car je perdrais de l'information.

Solution : je les **résume** avec `groupby()` + `agg()`. Par exemple, pour les articles d'une commande, je garde le nombre d'articles, le prix total et les frais de port totaux. J'obtiens alors une seule ligne par commande.

In [35]:
articles_par_commande = items.groupby("order_id").agg(
    nb_articles=("order_item_id", "count"),       # combien d'articles
    nb_produits=("product_id", "nunique"),        # combien de produits différents
    nb_vendeurs=("seller_id", "nunique"),         # combien de vendeurs différents
    montant_produits=("price", "sum"),            # prix total
    frais_port=("freight_value", "sum"),          # frais de port totaux
).reset_index()
articles_par_commande.head()

,order_id,nb_articles,nb_produits,nb_vendeurs,montant_produits,frais_port
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14


In [36]:
paiements_par_commande = payments.groupby("order_id").agg(
    montant_paye=("payment_value", "sum"),                    # total payé
    nb_paiements=("payment_sequential", "count"),             # en combien de paiements
    type_paiement=("payment_type", lambda s: s.mode()[0]),    # le moyen de paiement le plus utilisé
    nb_echeances_max=("payment_installments", "max"),         # payé en combien de fois au maximum
).reset_index()
paiements_par_commande.head()

,order_id,montant_paye,nb_paiements,type_paiement,nb_echeances_max
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,credit_card,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,credit_card,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,credit_card,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,credit_card,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,credit_card,3


Pour les avis, quand il y en a plusieurs, je garde la note du **plus récent** (je trie par date, puis je prends le dernier avec `last`).

In [37]:
avis_par_commande = (reviews_propre.sort_values("review_answer_timestamp")
                     .groupby("order_id").agg(
                         note=("review_score", "last"),
                         nb_avis=("review_id", "count"),
                         a_laisse_un_commentaire=("review_comment_message", lambda s: s.notna().any()),
                     ).reset_index())
avis_par_commande.head()

,order_id,note,nb_avis,a_laisse_un_commentaire
0,00010242fe8c5a6d1ba2dd792cb16214,5,1,True
1,00018f77f2f0320c557190d7a144bdd3,4,1,False
2,000229ec398224ef6ca0657da4fc703e,5,1,True
3,00024acbcdf0a6daa1e931b038114c75,4,1,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,5,1,True


Pour la géolocalisation, je garde un seul point par code postal : la **médiane** des coordonnées (moins sensible aux valeurs extrêmes que la moyenne) et la ville la plus fréquente.

In [38]:
geo_par_code_postal = geo_propre.groupby("geolocation_zip_code_prefix").agg(
    lat=("geolocation_lat", "median"),
    lng=("geolocation_lng", "median"),
    ville=("geolocation_city", lambda s: s.mode()[0]),
    etat=("geolocation_state", lambda s: s.mode()[0]),
).reset_index()
geo_par_code_postal.head()

,geolocation_zip_code_prefix,lat,lng,ville,etat
0,1001,-23.549951,-46.634027,sao paulo,SP
1,1002,-23.548228,-46.635247,sao paulo,SP
2,1003,-23.548977,-46.635313,sao paulo,SP
3,1004,-23.549550,-46.634771,sao paulo,SP
4,1005,-23.549763,-46.636100,sao paulo,SP


# 5. Table finale : une ligne par commande

Maintenant que tout est à une ligne par commande, je peux fusionner les tables avec `merge()`. J'utilise `how="left"` pour garder toutes les commandes, même celles qui n'ont pas d'avis ou de paiement.

In [40]:
commandes = (orders_propre
    .merge(customers, on="customer_id", how="left")
    .merge(articles_par_commande, on="order_id", how="left")
    .merge(paiements_par_commande, on="order_id", how="left")
    .merge(avis_par_commande, on="order_id", how="left"))

# Je vérifie qu'il n'y a pas de commande en double après les fusions
assert commandes.order_id.is_unique, "Attention : des commandes sont en double !"
print(commandes.shape)
commandes.head()

(99441, 27)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delai_livraison_jours,retard_jours,en_retard,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,nb_articles,nb_produits,nb_vendeurs,montant_produits,frais_port,montant_paye,nb_paiements,type_paiement,nb_echeances_max,note,nb_avis,a_laisse_un_commentaire
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,False,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,29.99,8.72,38.71,3.0,voucher,1.0,4.0,1.0,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,False,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,118.70,22.76,141.46,1.0,boleto,1.0,4.0,1.0,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,False,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,159.90,19.22,179.12,1.0,credit_card,3.0,5.0,1.0,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,False,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,45.00,27.20,72.20,1.0,credit_card,1.0,5.0,1.0,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,False,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,19.90,8.72,28.62,1.0,credit_card,1.0,5.0,1.0,False
